⏱️ **Time required:** ~5 minutes | **Type:** Hands-on quickstart

# LakeLogic — 5 Minute Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LakeLogic/LakeLogic/blob/main/examples/colab/00_quickstart.ipynb) [![View on GitHub](https://img.shields.io/badge/github-view_source-black?logo=github)](https://github.com/lakelogic/LakeLogic/blob/main/examples/colab/00_quickstart.ipynb)

One contract. One pipeline. Every row accounted for. Five minutes.

In [1]:
import subprocess
import sys
import importlib
import urllib.request
import os

if importlib.util.find_spec("lakelogic") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "-q", "lakelogic[polars]"])
if not os.path.exists("_setup.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/LakeLogic/LakeLogic/main/examples/colab/_setup.py", "_setup.py"
    )
import _setup as s
import lakelogic as ll

lakelogic v1.21.0 | Local | c:\_Personal\_SaaS\lakelogic\examples\colab


## The Problem

You have raw order data landing in your lake. Some rows have bad emails, negative amounts, or unknown statuses. You need to validate every row, quarantine the bad ones, and prove nothing was silently dropped — with zero custom Python logic.

## The Solution

In [2]:
contract = s.write_contract(
    """
version: 1.0.0
dataset: orders

info:
  title: E-Commerce Orders
  version: 1.0.0
  owner: data-team@company.com
  target_layer: silver

model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: customer_email
      type: string
      required: true
      pii: true
    - name: amount
      type: float
      required: true
    - name: currency
      type: string
    - name: status
      type: string
    - name: created_at
      type: string

transformations:
  - phase: "post"
    derive:
      field: "amount_gbp"
      sql: "CAST(CASE WHEN currency='USD' THEN amount*0.79 WHEN currency='EUR' THEN amount*0.86 ELSE amount END AS DECIMAL(10,2))"

quality:
  row_rules:
    - name: valid_email
      sql: "customer_email LIKE '%@%.%'"
    - name: positive_amount
      sql: "amount > 0"
    - name: valid_status
      sql: "status IN ('pending','shipped','delivered','returned')"
    - name: valid_currency
      sql: "currency IN ('GBP','USD','EUR')"
    - name: valid_order_id
      sql: "order_id > 0"

""",
    "00_quickstart_demo/orders_contract.yaml",
)

# Generate 1000 rows — 10% intentionally bad
source_df = ll.DataGenerator(contract).generate(rows=1000, invalid_ratio=0.10)

# Run the pipeline
proc = ll.DataProcessor(contract, engine="polars")
good, bad = proc.run(source_df)

2026-04-28 06:32:08.909 | INFO     | lakelogic.core.generator:generate:3416 - 📋 Generating data for: E-Commerce Orders
2026-04-28 06:32:08.909 | INFO     | lakelogic.core.generator:generate:3417 -    Records    : 900 valid + 100 invalid = 1,000 total
2026-04-28 06:32:08.910 | INFO     | lakelogic.core.generator:generate:3433 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-28 06:32:08.910 | INFO     | lakelogic.core.generator:generate:3448 -    Edge cases : Heuristic-only (no AI edge cases available)
2026-04-28 06:32:09.072 | INFO     | lakelogic.core.generator:generate:3482 -    Row generation complete: 1,000 records built
2026-04-28 06:32:09.073 | INFO     | lakelogic.core.generator:generate:3504 -    Test cases : 232 across 7 categories
2026-04-28 06:32:09.075 | INFO     | lakelogic.core.generator:generate:3506 -      NOT_NULL_VIOLATION               82 injections
2026-04-28 06:32:09.075 | INFO     | lakelogic.core.generator:generate:3506 -      EMPTY_STR

## The Proof

In [8]:
# Every row accounted for
s.assert_reconciliation(source_df, good, bad)

source=1000  good=886  bad=114
1000 == 886 + 114 -> True


In [9]:
# What was caught
print("Quarantined rows (sample):")
display(bad.head(10))

Quarantined rows (sample):


order_id,customer_email,amount,currency,status,created_at,_is_invalid,_test_case_types,amount_gbp,_lakelogic_errors,_lakelogic_categories,quarantine_state,quarantine_reprocessed
i64,str,f64,str,str,str,bool,str,"decimal[10,2]",list[str],list[str],str,bool
648,"""barbernicholas@example.org""",49.08,"""USD""","""INVALID_GHXR""","""2026-03-18T10:25:45.069374""",true,"""ACCEPTED_VALUE_VIOLATION""",38.77,"[""Rule failed: valid_status (status IN ('pending','shipped','delivered','returned'))""]","[""correctness""]","""active""",false
7584,"""ajohnson@example.net""",12.32,null,"""pending""","""2026-03-10T01:10:11.948471""",false,null,12.32,"[""Rule failed: valid_currency (currency IN ('GBP','USD','EUR'))""]","[""correctness""]","""active""",false
null,"""""",6.46,"""INVALID_HUUY""","""delivered""","""2026-03-05T22:55:50.066887""",true,"""ACCEPTED_VALUE_VIOLATION,EMPTY_STRING,NOT_NULL_VIOLATION""",6.46,"[""Rule failed: order_id_required (""order_id"" IS NOT NULL)"", ""Rule failed: valid_email (customer_email LIKE '%@%.%')"", … ""Rule failed: valid_order_id (order_id > 0)""]","[""completeness"", ""correctness"", … ""correctness""]","""active""",false
8341,"""rthompson@example.com""",null,"""INVALID_JSWQ""","""""","""2026-02-04T19:14:15.068261""",true,"""ACCEPTED_VALUE_VIOLATION,EMPTY_STRING,NOT_NULL_VIOLATION""",null,"[""Rule failed: amount_required (""amount"" IS NOT NULL)"", ""Rule failed: positive_amount (amount > 0)"", … ""Rule failed: valid_currency (currency IN ('GBP','USD','EUR'))""]","[""completeness"", ""correctness"", … ""correctness""]","""active""",false
4736,"""jacksonkristen@example.org""",28.63,null,"""shipped""","""2026-04-25T05:14:23.949605""",false,null,28.63,"[""Rule failed: valid_currency (currency IN ('GBP','USD','EUR'))""]","[""correctness""]","""active""",false
-999999999,null,3.47,"""USD""","""shipped""",null,true,"""BOUNDARY_VALUE,NOT_NULL_VIOLATION""",2.74,"[""Rule failed: customer_email_required (""customer_email"" IS NOT NULL)"", ""Rule failed: valid_email (customer_email LIKE '%@%.%')"", ""Rule failed: valid_order_id (order_id > 0)""]","[""completeness"", ""correctness"", ""correctness""]","""active""",false
8126,"""""",39.87,"""USD""","""INVALID_YPDX""","""2026-03-02T18:15:14.066887""",true,"""ACCEPTED_VALUE_VIOLATION,EMPTY_STRING""",31.50,"[""Rule failed: valid_email (customer_email LIKE '%@%.%')"", ""Rule failed: valid_status (status IN ('pending','shipped','delivered','returned'))""]","[""correctness"", ""correctness""]","""active""",false
6762,"""qconley@example.com""",121.68,null,"""shipped""","""2026-02-16T13:18:11.920899""",false,null,121.68,"[""Rule failed: valid_currency (currency IN ('GBP','USD','EUR'))""]","[""correctness""]","""active""",false
4151,"""lindsaylynch@example.net""",null,"""USD""","""pending""","""2033-10-13T06:32:09.064422""",true,"""NOT_NULL_VIOLATION,TEMPORAL_VIOLATION""",null,"[""Rule failed: amount_required (""amount"" IS NOT NULL)"", ""Rule failed: positive_amount (amount > 0)""]","[""completeness"", ""correctness""]","""active""",false


In [10]:
# What was good
print("Valid rows (sample):")
display(good.head(10))

Valid rows (sample):


order_id,customer_email,amount,currency,status,created_at,_is_invalid,_test_case_types,amount_gbp
i64,str,f64,str,str,str,bool,str,"decimal[10,2]"
738,"""simmonshailey@example.net""",397.87,"""GBP""","""delivered""","""2026-03-29T00:11:16.991998""",false,null,397.87
1376,"""andersonjustin@example.net""",15.94,"""GBP""","""delivered""","""2026-03-16T00:22:53.945857""",false,null,15.94
6483,"""rachel54@example.org""",43.25,"""USD""","""returned""","""2026-02-01T01:15:26.025272""",false,null,34.17
3829,"""michael53@example.net""",337.81,"""USD""","""returned""","""2026-02-09T17:27:55.929629""",false,null,266.87
9642,"""tdixon@example.org""",24.93,"""EUR""","""delivered""","""2026-03-27T04:38:16.007147""",false,null,21.44
5976,"""nchambers@example.org""",66.54,"""USD""","""delivered""","""2026-03-18T00:06:25.002314""",false,null,52.57
5863,"""orogers@example.com""",11.12,"""GBP""","""pending""","""2026-01-29T21:41:15.930281""",false,null,11.12
3462,"""smithluke@example.net""",21.39,"""GBP""","""shipped""","""2026-04-24T06:55:59.991998""",false,null,21.39
1764,"""buchananpamela@example.net""",8.72,"""EUR""","""shipped""","""2026-04-17T12:00:35.961661""",false,null,7.50


In [11]:
# Full audit trail
s.print_report(proc)

Run ID      : b2253cb4-568f-48d1-a47f-39654370cf49
Timestamp   : 2026-04-28T05:32:09+00:00
Source      : 1000
Good        : 886
Quarantined : 114

Rule failures:
  valid_status: 40 rows
  valid_currency: 54 rows
  order_id_required: 13 rows
  valid_email: 42 rows
  valid_order_id: 38 rows
  amount_required: 26 rows
  positive_amount: 40 rows
  customer_email_required: 15 rows


{'run_id': 'b2253cb4-568f-48d1-a47f-39654370cf49',
 'pipeline_run_id': None,
 'engine': 'polars',
 'contract': 'E-Commerce Orders',
 'contract_file_name': None,
 'contract_version': '1.0.0',
 'stage': 'default',
 'dataset': 'orders',
 'domain': None,
 'system': None,
 'environment': 'local',
 'data_layer': 'silver',
 'source_path': None,
 'source_files': [],
 'max_source_mtime': None,
 'timestamp': '2026-04-28T05:32:09+00:00',
 'counts': {'source': 1000,
  'total': 1000,
  'good': 886,
  'quarantined': 114,
  'quarantine_ratio': 0.114,
  'pre_transform_dropped': 0},
 'dataset_rules': [],
 'slos': {},
 'row_rule_failures': [{'name': 'valid_status',
   'sql': "status IN ('pending','shipped','delivered','returned')",
   'message': "Rule failed: valid_status (status IN ('pending','shipped','delivered','returned'))",
   'count': 40},
  {'name': 'valid_currency',
   'sql': "currency IN ('GBP','USD','EUR')",
   'message': "Rule failed: valid_currency (currency IN ('GBP','USD','EUR'))",
   'co

## What You Just Saw

- **One YAML contract** defined schema, quality rules, and a derived column
- **100% reconciliation** — source == good + bad, every row accounted for
- **Automatic audit trail** — run ID, timestamp, per-rule failure counts

---
## Go Deeper — Explore by Capability

Each notebook below maps to a pillar of LakeLogic's [Technical Capabilities](https://lakelogic.github.io/LakeLogic/#technical-capabilities):

| # | Notebook | What You'll See |
|---|---|---|
| 🛡️ | **[Data Quality & Trust](01_data_quality_trust.ipynb)** | Reconciliation proofs, Pydantic validation, SQL-first rules, SLO monitoring |
| 📜 | **[Compliance & Governance](02_compliance_governance.ipynb)** | GDPR erasure in 2 lines, automatic lineage, cost intelligence |
| ⚡ | **[Engine & Scale](03_engine_scale.ipynb)** | Same contract on Polars & DuckDB, incremental processing, dry run |
| 🔧 | **[Developer Experience](04_developer_experience.ipynb)** | Structured diagnostics, DDL generation, surgical resets, multi-channel alerts |
| 🧬 | **[Data Generation & AI](05_data_generation_ai.ipynb)** | Synthetic data, referential integrity, edge case injection, contract inference |
| 🔌 | **[Integrations](06_integrations.ipynb)** | dbt adapter, dlt sources, contract-driven quality gates on arrival |

> **Each notebook is self-contained** — pick the capability that matters most to you and run it independently.